# 01 · Limpieza de datos

**Proyecto:** LINE — Auditor Médico Digital · Health & Life IPS SAS
**Capstone Samsung Innovation Campus 2025**

Este notebook limpia las 5 tablas originales del ciclo de facturación médica y
produce versiones consistentes en `data/processed/`.

| Entrada | Salida |
|---|---|
| `data/raw/01_pacientes.csv` … `05_cruce_validacion.csv` | `data/processed/*_clean.csv` |
| | `outputs/reports/limpieza_reporte.{md,json}` |

**Reglas de oro:**
1. `data/raw/` nunca se modifica.
2. Los nulos *semánticos* del cruce se conservan: `id_prefactura` nulo significa
   **NO_FACTURADO** (fuga de ingreso) e `id_detalle_hc` nulo significa
   **SIN_SOPORTE_CLINICO**. Borrarlos destruiría exactamente los casos que el
   auditor debe detectar.

## 0 · Configuración (celda autocontenida)
Detecta la raíz del proyecto, crea carpetas de salida y define utilidades de
escritura robusta. La carpeta puede vivir bajo OneDrive, que bloquea archivos
intermitentemente mientras sincroniza — por eso toda escritura es atómica
(`.tmp` + `os.replace`) con reintentos.

In [6]:
from pathlib import Path
import json
import os
import time
import unicodedata

import numpy as np
import pandas as pd

try:  # ejecutando como script .py desde src/
    ROOT = Path(__file__).resolve().parents[1]
except NameError:  # ejecutando como notebook desde notebooks/
    ROOT = Path.cwd()
    if ROOT.name in ("notebooks", "src"):
        ROOT = ROOT.parent

DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
OUT_REP = ROOT / "outputs" / "reports"
for _d in (DATA_PROC, OUT_REP):
    _d.mkdir(parents=True, exist_ok=True)

try:
    display  # noqa: B018 — definido por IPython en notebooks
except NameError:
    display = print


def reintentar(fn, intentos=6, espera=0.5):
    """Reintenta fn() ante OSError (bloqueos transitorios del sync de OneDrive)."""
    for _i in range(intentos):
        try:
            return fn()
        except OSError:
            if _i == intentos - 1:
                raise
            time.sleep(espera * (2 ** _i))


def to_csv_seguro(df, path, **kw):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    reintentar(lambda: df.to_csv(tmp, **kw))
    reintentar(lambda: os.replace(tmp, path))


def write_text_seguro(path, texto, encoding="utf-8"):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    reintentar(lambda: tmp.write_text(texto, encoding=encoding))
    reintentar(lambda: os.replace(tmp, path))


FILES = {
    "pacientes": "01_pacientes.csv",
    "atenciones": "02_atenciones.csv",
    "hc_detalle": "03_historia_clinica_detalle.csv",
    "prefactura": "04_prefactura.csv",
    "cruce": "05_cruce_validacion.csv",
}

_faltan = [str(DATA_RAW / f) for f in FILES.values() if not (DATA_RAW / f).exists()]
assert not _faltan, (
    "FALTAN INSUMOS para este notebook:\n  - " + "\n  - ".join(_faltan)
    + "\n→ Copia los 5 CSVs originales de Health & Life a la carpeta data/raw/."
)
print(f"Proyecto: {ROOT.name} | 5 CSVs de entrada encontrados en data/raw/")

Proyecto: LINE | 5 CSVs de entrada encontrados en data/raw/


## 1 · Carga y normalización básica
- Nombres de columnas a minúsculas, sin tildes ni espacios.
- Espacios sobrantes eliminados en celdas de texto; cadenas vacías → nulo.

In [7]:
def normalizar_columnas(df):
    def norm(c):
        c = c.strip().lower().replace(" ", "_")
        return unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode()
    df.columns = [norm(c) for c in df.columns]
    return df


from pandas.api.types import is_string_dtype

def limpiar_strings(df):
    # Detectar columnas con tipo string (object o pandas StringDtype) de forma segura
    str_cols = [c for c in df.columns if is_string_dtype(df[c].dtype)]
    for c in str_cols:
        # Convertir a pandas StringDtype para preservar NA y usar .str
        df[c] = df[c].astype("string").str.strip()
    if str_cols:
        df.loc[:, str_cols] = df.loc[:, str_cols].replace("", pd.NA)
    return df


data = {}
for name, fname in FILES.items():
    df = pd.read_csv(DATA_RAW / fname, encoding="utf-8", dtype=str)
    data[name] = limpiar_strings(normalizar_columnas(df))
    print(f"{name:12s} -> {data[name].shape[0]:>5,} filas x {data[name].shape[1]} cols")

pacientes    ->   300 filas x 7 cols
atenciones   -> 1,200 filas x 9 cols
hc_detalle   -> 3,058 filas x 9 cols
prefactura   -> 2,974 filas x 10 cols
cruce        -> 3,126 filas x 8 cols


## 2 · Limpieza por tabla
Casteo de tipos, chequeo de duplicados, rangos y llaves foráneas.
Todo hallazgo queda en el diccionario `reporte`.

In [8]:
reporte = {}


def check_cups(serie):
    """Formato CUPS interno: 6 dígitos numéricos."""
    return serie.dropna().astype(str).str.fullmatch(r"\d{6}")


def resumen_tabla(df, pk=None):
    info = {
        "filas": int(len(df)),
        "columnas": int(df.shape[1]),
        "duplicados_fila_completa": int(df.duplicated().sum()),
        "nulos_por_columna": {k: int(v) for k, v in df.isna().sum().items() if v > 0},
    }
    if pk and pk in df.columns:
        info["pk"] = pk
        info["pk_duplicada"] = int(df[pk].duplicated().sum())
    return info


# --- pacientes ---
p = data["pacientes"]
p["edad"] = pd.to_numeric(p["edad"], errors="coerce")
reporte["pacientes"] = resumen_tabla(p, "id_paciente")
reporte["pacientes"]["edad_fuera_rango_0_120"] = int(((p["edad"] < 0) | (p["edad"] > 120)).sum())

# --- atenciones ---
a = data["atenciones"]
a["fecha_atencion"] = pd.to_datetime(a["fecha_atencion"], errors="coerce")
reporte["atenciones"] = resumen_tabla(a, "id_atencion")
reporte["atenciones"]["fechas_invalidas"] = int(a["fecha_atencion"].isna().sum())
reporte["atenciones"]["rango_fechas"] = [
    str(a["fecha_atencion"].min().date()), str(a["fecha_atencion"].max().date())]
cie_ok = a["diagnostico_principal_cie10"].dropna().str.fullmatch(r"[A-Z]\d{2,3}\d?")
reporte["atenciones"]["cie10_mal_formateados"] = int((~cie_ok).sum())
reporte["atenciones"]["fk_pacientes_rotas"] = int((~a["id_paciente"].isin(p["id_paciente"])).sum())

# --- hc_detalle ---
h = data["hc_detalle"].rename(columns={"id_detalle": "id_detalle_hc"})
h["cantidad_realizada"] = pd.to_numeric(h["cantidad_realizada"], errors="coerce")
h["fecha_registro"] = pd.to_datetime(h["fecha_registro"], errors="coerce")
reporte["hc_detalle"] = resumen_tabla(h, "id_detalle_hc")
reporte["hc_detalle"]["cantidades_negativas"] = int((h["cantidad_realizada"] < 0).sum())
reporte["hc_detalle"]["cantidades_nulas_a_0"] = int(h["cantidad_realizada"].isna().sum())
h["cantidad_realizada"] = h["cantidad_realizada"].fillna(0).astype(int)
reporte["hc_detalle"]["cups_mal_formateados"] = int((~check_cups(h["codigo_cups"])).sum())
reporte["hc_detalle"]["fk_atenciones_rotas"] = int((~h["id_atencion"].isin(a["id_atencion"])).sum())
reporte["hc_detalle"]["valores_soporte_clinico"] = h["soporte_clinico"].value_counts(dropna=False).to_dict()

# --- prefactura ---
f = data["prefactura"]
for col in ["cantidad_facturada", "valor_unitario", "valor_total"]:
    f[col] = pd.to_numeric(f[col], errors="coerce")
f["fecha_facturacion"] = pd.to_datetime(f["fecha_facturacion"], errors="coerce")
reporte["prefactura"] = resumen_tabla(f, "id_prefactura")
reporte["prefactura"]["cantidades_negativas"] = int((f["cantidad_facturada"] < 0).sum())
reporte["prefactura"]["valores_negativos"] = int(((f["valor_unitario"] < 0) | (f["valor_total"] < 0)).sum())
calc = f["cantidad_facturada"] * f["valor_unitario"]
incoherentes = (calc - f["valor_total"]).abs() > 1
reporte["prefactura"]["valor_total_incoherente"] = int(incoherentes.fillna(False).sum())
reporte["prefactura"]["cups_mal_formateados"] = int((~check_cups(f["codigo_cups_facturado"])).sum())
reporte["prefactura"]["fk_atenciones_rotas"] = int((~f["id_atencion"].isin(a["id_atencion"])).sum())
reporte["prefactura"]["fk_pacientes_rotas"] = int((~f["id_paciente"].isin(p["id_paciente"])).sum())

# --- cruce (tabla de validación con el target) ---
c = data["cruce"]
reporte["cruce"] = resumen_tabla(c, "id_cruce")
reporte["cruce"]["nulos_semanticos"] = {
    "id_prefactura_nulo": {
        "n": int(c["id_prefactura"].isna().sum()),
        "tipo_alerta_asociado": c.loc[c["id_prefactura"].isna(), "tipo_alerta"].value_counts().to_dict(),
    },
    "id_detalle_hc_nulo": {
        "n": int(c["id_detalle_hc"].isna().sum()),
        "tipo_alerta_asociado": c.loc[c["id_detalle_hc"].isna(), "tipo_alerta"].value_counts().to_dict(),
    },
    "decision": "SE CONSERVAN: codifican NO_FACTURADO y SIN_SOPORTE_CLINICO",
}
reporte["cruce"]["distribucion_resultado"] = c["resultado"].value_counts().to_dict()
reporte["cruce"]["distribucion_tipo_alerta"] = c["tipo_alerta"].value_counts().to_dict()
reporte["cruce"]["fk_atencion_rotas"] = int((~c["id_atencion"].isin(a["id_atencion"])).sum())
reporte["cruce"]["fk_prefactura_rotas"] = int((~c["id_prefactura"].dropna().isin(f["id_prefactura"])).sum())
reporte["cruce"]["fk_detalle_hc_rotas"] = int((~c["id_detalle_hc"].dropna().isin(h["id_detalle_hc"])).sum())

for tabla, info in reporte.items():
    print(f"{tabla}: {info['filas']:,} filas | dup={info['duplicados_fila_completa']}"
          f" | pk_dup={info.get('pk_duplicada', 'n/a')}")

pacientes: 300 filas | dup=0 | pk_dup=0
atenciones: 1,200 filas | dup=0 | pk_dup=0
hc_detalle: 3,058 filas | dup=0 | pk_dup=0
prefactura: 2,974 filas | dup=0 | pk_dup=0
cruce: 3,126 filas | dup=0 | pk_dup=0


## 3 · Vista rápida de los nulos semánticos
La correspondencia entre nulos y tipo de alerta debe ser del 100 % — es la
evidencia de que son *casos reales*, no errores de captura.

In [9]:
display(pd.DataFrame(reporte["cruce"]["nulos_semanticos"]).T)

,n,tipo_alerta_asociado
id_prefactura_nulo,152,{'NO_FACTURADO': 152}
id_detalle_hc_nulo,70,{'SIN_SOPORTE_CLINICO': 70}
decision,SE CONSERVAN: codifican NO_FACTURADO y SIN_SOP...,SE CONSERVAN: codifican NO_FACTURADO y SIN_SOP...


## 4 · Exportar tablas limpias y reporte
- CSVs limpios → `data/processed/`
- Reporte de limpieza → `outputs/reports/limpieza_reporte.{json,md}`

In [10]:
to_csv_seguro(p, DATA_PROC / "pacientes_clean.csv", index=False)
to_csv_seguro(a, DATA_PROC / "atenciones_clean.csv", index=False)
to_csv_seguro(h, DATA_PROC / "hc_detalle_clean.csv", index=False)
to_csv_seguro(f, DATA_PROC / "prefactura_clean.csv", index=False)
to_csv_seguro(c, DATA_PROC / "cruce_clean.csv", index=False)

write_text_seguro(OUT_REP / "limpieza_reporte.json",
                  json.dumps(reporte, ensure_ascii=False, indent=2, default=str))

lineas = ["# Reporte de limpieza — datasets 1-5", ""]
for tabla, info in reporte.items():
    lineas.append(f"## {tabla}")
    lineas += [f"- **{k}**: {v}" for k, v in info.items()]
    lineas.append("")
write_text_seguro(OUT_REP / "limpieza_reporte.md", "\n".join(lineas))

print("Limpieza completada.")
print(f"  Tablas limpias : {DATA_PROC}")
print(f"  Reporte        : {OUT_REP / 'limpieza_reporte.md'}")

Limpieza completada.
  Tablas limpias : c:\Users\MIGI\Desktop\LINE\data\processed
  Reporte        : c:\Users\MIGI\Desktop\LINE\outputs\reports\limpieza_reporte.md


## ✅ Verificación de cierre
Este notebook terminó bien si existen los 5 `*_clean.csv` en `data/processed/`
y `limpieza_reporte.md` en `outputs/reports/`, y si arriba se reportan
**0 duplicados de PK**.

**Ojo con las llaves foráneas:** el resumen impreso no las muestra, pero el
`limpieza_reporte.json` sí las registra. Con los datos actuales hay **2 filas**
en `hc_detalle` (`DET-0003057`, `DET-0003058`) que apuntan a la atención
`ATN-JEF-000001`, inexistente en `atenciones` (`fk_atenciones_rotas: 2`).

Es un **dato de prueba interno del equipo** — la atención demo del paciente
`1005711681` que usa el aplicativo LINE —, **no un hallazgo para Health & Life**.
Se **conserva** para no alterar los insumos de `data/raw/`, pero hay que tenerlo
presente: es la causa del "1.201 atenciones" del notebook 02 y de los 2 códigos
`SOLO_HC` del notebook 03. En una carga con datos reales debe excluirse junto
con el resto de los datos de prueba.
